# Extract raster values at points

- **`sample(points)`** — read the raster value at each point location (e.g. gauge stations).
- **`extract()`** — pull every valid (non-no-data) cell value into a flat array, handy for
  histograms or training samples.

## Setup

In [ ]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

In [ ]:
from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
gauges = FeatureCollection.read_file(str(DATA / 'coello-gauges.geojson'))
ds.shape, ds.epsg, (len(gauges), gauges.epsg)

## Sample at point locations — `sample`

Returns an array of shape `(bands, n_points)` — one value per band per point.

In [ ]:
pts = gauges if gauges.epsg == ds.epsg else FeatureCollection(gauges.to_crs(ds.epsg))
values = ds.sample(pts)
values.shape, np.asarray(values).ravel()

## All valid cell values — `extract`

A 1-D array of the cells that are not no-data.

In [ ]:
cells = ds.extract()
cells.shape, float(cells.min()), float(cells.max())

## Notes

- `sample` accepts a `FeatureCollection`, a GeoDataFrame, or a plain DataFrame of x/y.
- `bands=` on `sample` selects which band(s) to read.
- See also: [Zonal statistics](zonal-statistics.ipynb).